Дорогой студент!

В данном домашнем задании предоставляется возможность поработать с задачей по распознаванию позитивных и негативных отзывы людей по автомобилю Tesla. База для обучения содержит два текстовых файла с рядом строчных отзывов с мнением людей об автомобиле Tesla, соответственно негативного и позитивного содержания. Ссылка на скачивание базы уже включена в ноутбук задания.


Необходимо выполнить следующие действия:

  1. Загрузите саму базу по ссылке и подговьте файлы базы для обработки.
  2. Создайте обучающую и проверочную выборки, обратив особое внимание на балансировку базы: количество примеров каждого класса должно быть примерно одного порядка.
  3. Подготовьте выборки для обучения и обучите сеть. Добейтесь результата точности сети в 85-90% на проверочной выборке.
   


## 1. Импорт библиотек

In [ ]:
# Работа с массивами данных
import numpy as np

# Работа с таблицами
import pandas as pd

# Отрисовка графиков
import matplotlib.pyplot as plt

# Функции-утилиты для работы с категориальными данными
from tensorflow.keras import utils

# Класс для конструирования последовательной модели нейронной сети
from tensorflow.keras.models import Sequential

# Основные слои
from tensorflow.keras.layers import Dense, Dropout, SpatialDropout1D, BatchNormalization, Embedding, Flatten, Activation

# Токенизатор для преобразование текстов в последовательности
from tensorflow.keras.preprocessing.text import Tokenizer

# Заполнение последовательностей до определенной длины
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Загрузка датасетов из облака google
import gdown

# Для работы с файлами в Colaboratory
import os

# Отрисовка графиков
import matplotlib.pyplot as plt

%matplotlib inline

## 2. Скачивание датасета с отзывами о Tesla

In [ ]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/tesla.zip', None, quiet=True)

'tesla.zip'

## 3. Распаковка архива и просмотр файлов

In [ ]:
# Распаковка архива в папку writers
!unzip -qo tesla.zip -d tesla/

# Просмотр содержимого папки
!ls tesla

'Негативный отзыв.txt'	'Позитивный отзыв.txt'


## 4. Функция чтения текстовых файлов и объявление классов

In [ ]:
# Объявляем функции для чтения файла. На вход отправляем путь к файлу
def read_text(file_name):

  # Задаем открытие нужного файла в режиме чтения
  read_file = open(file_name, 'r')

  # Читаем текст
  text = read_file.read()

  # Переносы строки переводим в пробелы
  text = text.replace("\n", " ")

  # Возвращаем текст файла
  return text

# Объявляем интересующие нас классы
class_names = ["Негативный отзыв", "Позитивный отзыв"]

# Считаем количество классов
num_classes = len(class_names)

## 5. Загрузка отзывов в список texts_list

In [ ]:
import os
# Создаём список под тексты для обучающей выборки
texts_list = []

# Циклом проводим итерацию по текстовым файлам в папке отзывов
for j in os.listdir('/content/tesla/'):

  # Добавляем каждый файл в общий список для выборки
        texts_list.append(read_text('/content/tesla/' + j))

        # Выводим на экран сообщение о добавлении файла
        print(j, 'добавлен в обучающую выборку')

Негативный отзыв.txt добавлен в обучающую выборку
Позитивный отзыв.txt добавлен в обучающую выборку


## 6. Вывод размеров загруженных текстов в символах

In [ ]:
# Узнаем объём каждого текста в словах и символах
texts_len = [len(text) for text in texts_list]

# Устанавливаем "счётчик" номера текста
t_num = 0

# Выводим на экран  информационное сообщение
print(f'Размеры текстов по порядку (в токенах):')

# Циклом проводим итерацию по списку с объёмами текстов
for text_len in texts_len:

  # Запускаем "счётчик" номера текста
  t_num += 1

  # Выводим на экран сообщение о номере и объёме текста
  print(f'Текст №{t_num}: {text_len}')

Размеры текстов по порядку (в токенах):
Текст №1: 134535
Текст №2: 213381


Далее рассчитаем, сколько символов составит 80% объёма каждого текста, чтобы по полученному индексу отделить эти 80% на обучающую и оставшиеся 20% на проверочную выборку. Эти значения необходимы для подготовки деления на выборки слайсингом по индексу.

## 7. Вычисление объёма 80% от каждого текста

In [ ]:
# Создаём список с вложенным циклом по длинам текстов, где i - 100% текста, i/5 - 20% текста
train_len_shares = [(i - round(i/5)) for i in texts_len]

# Устанавливаем "счётчик" номера текста
t_num = 0

# Циклом проводим итерацию по списку с объёмами текстов равными 80% от исходных
for train_len_share in train_len_shares:

  # Запускаем "счётчик" номера текста
  t_num += 1

  # Выводим на экран сообщение о номере и объёме текста в 80% от исходного
  print(f'Доля 80% от текста №{t_num}: {train_len_share} символов')

Доля 80% от текста №1: 107628 символов
Доля 80% от текста №2: 170705 символов


Импортируем функцию **chain()** для добавления текстов в каждую выборку.

---
 Дополнительная информация: ([База знаний УИИ  - **"Методы работы со списками: функция chain( )**"](https://colab.research.google.com/drive/1KJKg_WYD8Vq63cciOMBEEAhFpyPv0A0V?usp=sharing/))

---

Производём нарезку (метод слайсинга) по полученному ранее индексу для формирования текстов отдельно для обучающей(80%) и проверочной(20%) выборок:

## 8. Балансировка выборки (разбиение на фрагменты, train/test)

In [ ]:
from itertools import chain
import numpy as np
from sklearn.model_selection import train_test_split

print("Исходные данные:")
print(f"  Негативных отзывов (символов): {len(texts_list[0])}")
print(f"  Позитивных отзывов (символов): {len(texts_list[1])}")

CHUNK_SIZE = 500
STEP = 250

def split_text_into_chunks(text, chunk_size, step):
    """Разбивает текст на перекрывающиеся фрагменты"""
    chunks = []
    for i in range(0, len(text) - chunk_size + 1, step):
        chunks.append(text[i:i + chunk_size])
    return chunks

train_ratio = 0.8

neg_len = len(texts_list[0])
pos_len = len(texts_list[1])

neg_train_text = texts_list[0][:int(neg_len * train_ratio)]
neg_test_text = texts_list[0][int(neg_len * train_ratio):]

pos_train_text = texts_list[1][:int(pos_len * train_ratio)]
pos_test_text = texts_list[1][int(pos_len * train_ratio):]

print(f"\nРазмеры частей:")
print(f"  Негативный train: {len(neg_train_text)} символов")
print(f"  Негативный test: {len(neg_test_text)} символов")
print(f"  Позитивный train: {len(pos_train_text)} символов")
print(f"  Позитивный test: {len(pos_test_text)} символов")

neg_train_chunks = split_text_into_chunks(neg_train_text, CHUNK_SIZE, STEP)
neg_test_chunks = split_text_into_chunks(neg_test_text, CHUNK_SIZE, STEP)

pos_train_chunks = split_text_into_chunks(pos_train_text, CHUNK_SIZE, STEP)
pos_test_chunks = split_text_into_chunks(pos_test_text, CHUNK_SIZE, STEP)

print(f"\nПосле разбиения на фрагменты:")
print(f"  Негативных train: {len(neg_train_chunks)} фрагментов")
print(f"  Негативных test: {len(neg_test_chunks)} фрагментов")
print(f"  Позитивных train: {len(pos_train_chunks)} фрагментов")
print(f"  Позитивных test: {len(pos_test_chunks)} фрагментов")

X_train_texts = neg_train_chunks + pos_train_chunks
y_train = [0] * len(neg_train_chunks) + [1] * len(pos_train_chunks)

X_test_texts = neg_test_chunks + pos_test_chunks
y_test = [0] * len(neg_test_chunks) + [1] * len(pos_test_chunks)

print(f"\nИтоговая выборка:")
print(f"  Обучающая: {len(X_train_texts)} примеров")
print(f"    Негативных: {y_train.count(0)}")
print(f"    Позитивных: {y_train.count(1)}")
print(f"  Тестовая: {len(X_test_texts)} примеров")
print(f"    Негативных: {y_test.count(0)}")
print(f"    Позитивных: {y_test.count(1)}")

Исходные данные:
  Негативных отзывов (символов): 134535
  Позитивных отзывов (символов): 213381

Размеры частей:
  Негативный train: 107628 символов
  Негативный test: 26907 символов
  Позитивный train: 170704 символов
  Позитивный test: 42677 символов

После разбиения на фрагменты:
  Негативных train: 429 фрагментов
  Негативных test: 106 фрагментов
  Позитивных train: 681 фрагментов
  Позитивных test: 169 фрагментов

Итоговая выборка:
  Обучающая: 1110 примеров
    Негативных: 429
    Позитивных: 681
  Тестовая: 275 примеров
    Негативных: 106
    Позитивных: 169


## 9. Bag of Words (BoW) векторизация текстов

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000,
                             ngram_range=(1, 2),
                             min_df=2,
                             max_df=0.8)
X_train_bow = vectorizer.fit_transform(X_train_texts)
X_test_bow = vectorizer.transform(X_test_texts)

X_train_bow = X_train_bow.toarray()
X_test_bow = X_test_bow.toarray()

y_train = np.array(y_train)
y_test = np.array(y_test)

print(f"Форма BoW обучающей выборки: {X_train_bow.shape}")
print(f"Форма BoW тестовой выборки: {X_test_bow.shape}")
print(f"Размер словаря: {len(vectorizer.get_feature_names_out())}")

Форма BoW обучающей выборки: (1110, 5000)
Форма BoW тестовой выборки: (275, 5000)
Размер словаря: 5000


## 10. Создание и обучение полносвязной нейросети на BoW

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Веса классов: {class_weight_dict}")

model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train_bow.shape[1],)))
model.add(Dropout(0.5))
model.add(BatchNormalization())

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(BatchNormalization())

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.4))

model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max')

history = model.fit(
    X_train_bow, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_bow, y_test),
    callbacks=[early_stop],
    class_weight=class_weight_dict,
    verbose=1
)

Веса классов: {0: np.float64(1.2937062937062938), 1: np.float64(0.8149779735682819)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 9s 91ms/step - accuracy: 0.5748 - loss: 0.8193 - val_accuracy: 0.5491 - val_loss: 0.6866
Epoch 2/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6234 - loss: 0.6628 - val_accuracy: 0.6473 - val_loss: 0.6655
Epoch 3/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7252 - loss: 0.5364 - val_accuracy: 0.7382 - val_loss: 0.6182
Epoch 4/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8099 - loss: 0.4147 - val_accuracy: 0.8036 - val_loss: 0.5390
Epoch 5/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8441 - loss: 0.3210 - val_accuracy: 0.8327 - val_loss: 0.4554
Epoch 6/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9216 - loss: 0.2115 - val_accuracy: 0.8400 - val_loss: 0.3987
Epoch 7/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9523 - loss: 0.1557 - val_accuracy: 0.8400 - val_loss: 0.3637
Epoch 8/50
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9550 - loss: 0.1264 - val_accuracy: 0.8436 - val_loss

## 11. Оценка точности на тестовой выборке

In [ ]:
# Оценка на тестовой выборке
test_loss, test_acc = model.evaluate(X_test_bow, y_test, verbose=0)

print(f"Точность на тестовой выборке: {test_acc*100:.2f}%")


Точность на тестовой выборке: 84.73%
